# Build Master Player Database

Merges ratings using **name + season + league** to distinguish between players with the same name.

**Output:** `data/processed/player_db.csv`

In [295]:
import pandas as pd
import numpy as np
import os
import unicodedata
import warnings
warnings.filterwarnings('ignore')

In [296]:
RAW_DIR = os.path.join('..', 'data', 'raw')
OUTPUT_PATH = os.path.join('..', 'data', 'processed', 'player_db.csv')

LEAGUE_MAP = {
    'de Bundesliga': 'Bundesliga',
    'fr Ligue 1': 'Ligue 1',
    'es La Liga': 'La Liga',
    'eng Premier League': 'Premier League',
    'it Serie A': 'Serie A',
}

LEAGUE_FOLDERS = {
    'BLPlayerRat': ('BL', 'Bundesliga'),
    'L1PlayerRat': ('L1', 'Ligue 1'),
    'LLPlayerRat': ('LL', 'La Liga'),
    'PLPlayerRat': ('PL', 'Premier League'),
    'SAPlayerRat': ('SA', 'Serie A'),
}

In [297]:
def normalise_name(name):
    if pd.isna(name):
        return ''
    name = str(name).strip().lower()
    name = unicodedata.normalize('NFD', name)
    return ''.join(c for c in name if unicodedata.category(c) != 'Mn')

def convert_position_format(pos_str):
    if pd.isna(pos_str) or str(pos_str).strip() == '':
        return np.nan
    return str(pos_str).strip().replace(',', '/')

In [298]:
def clean_player_stats_rows(df):
    df = df.copy()
    
    df = df.drop(columns=['Rk'], errors='ignore')
    df.insert(0, 'rank', range(1, len(df) + 1))
    
    if 'Player' in df.columns:
        df = df[df['Player'] != 'Player'].copy()
    
    mask = df.apply(lambda col: col.astype(str).str.strip() == '90s', axis=0).any(axis=1)
    df = df[~mask].reset_index(drop=True)
    
    if 'Pos' in df.columns:
        df['Pos'] = df['Pos'].apply(convert_position_format)
    
    if 'Min' in df.columns:
        df['Min'] = df['Min'].astype(str).str.replace(',', '').str.strip()
        df['Min'] = pd.to_numeric(df['Min'], errors='coerce')
    
    if all(c in df.columns for c in ['Player', 'Age', 'Min']):
        df = (
            df.sort_values('Min', ascending=False, na_position='last')
              .drop_duplicates(subset=['Player', 'Age'], keep='first')
              .reset_index(drop=True)
        )
    
    df['rank'] = range(1, len(df) + 1)
    return df

In [299]:
def load_player_stats_files():
    season_map = {
        'player_stats_20-21.csv': '2020-2021',
        'player_stats_21-22.csv': '2021-2022',
        'player_stats_22-23.csv': '2022-2023',
        'player_stats_23-24.csv': '2023-2024',
        'player_stats_24-25.csv': '2024-2025',
    }
    
    all_stats = []
    
    for filename, season in season_map.items():
        path = os.path.join(RAW_DIR, 'PlayerStats', filename)
        if not os.path.exists(path):
            continue
        
        print(f"  Loading {filename}...")
        df = pd.read_csv(path)
        df = clean_player_stats_rows(df)
        df['season'] = season
        
        df.rename(columns={
            'Player': 'name',
            'Pos': 'position',
            'Squad': 'club',
            'Comp': 'league_comp',
            'MP': 'appearances',
            'Min': 'minutes',
            'Gls': 'goals',
            'Ast': 'assists',
        }, inplace=True, errors='ignore')
        
        all_stats.append(df)
        print(f"    → {len(df):,} rows")
    
    return pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()

print("Loading stats...\n")
player_stats = load_player_stats_files()

if not player_stats.empty:
    print(f"\nTotal: {len(player_stats):,} | Players: {player_stats['name'].nunique():,}")
    display(player_stats.head())

Loading stats...

  Loading player_stats_20-21.csv...
    → 2,704 rows
  Loading player_stats_21-22.csv...
    → 2,791 rows
  Loading player_stats_22-23.csv...
    → 2,724 rows
  Loading player_stats_23-24.csv...
    → 2,710 rows
  Loading player_stats_24-25.csv...
    → 2,707 rows

Total: 13,636 | Players: 5,595


,rank,name,Nation,position,club,league_comp,Age,Born,appearances,Starts,...,PK,PKatt,CrdY,CrdR,Gls_90,Ast_90,G+A_90,G-PK_90,G+A-PK_90,season
0,1,Thibaut Courtois,be BEL,GK,Real Madrid,es La Liga,28,1992,38,38,...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,2020-2021
1,2,Alban Lafont,ci CIV,GK,Nantes,fr Ligue 1,21,1999,38,38,...,0,0,2,0,0.00,0.00,0.00,0.00,0.00,2020-2021
2,3,Kasper Schmeichel,dk DEN,GK,Leicester City,eng Premier League,33,1986,38,38,...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,2020-2021
3,4,Baptiste Reynet,fr FRA,GK,Nîmes,fr Ligue 1,29,1990,38,38,...,0,0,3,0,0.00,0.00,0.00,0.00,0.00,2020-2021
4,5,Emiliano Martínez,ar ARG,GK,Aston Villa,eng Premier League,27,1992,38,38,...,0,0,1,0,0.00,0.00,0.00,0.00,0.00,2020-2021


In [300]:
def load_player_ratings():
    year_map = {
        '2021': '2020-2021',
        '2122': '2021-2022',
        '2223': '2022-2023',
        '2324': '2023-2024',
        '2425': '2024-2025',
    }
    
    all_ratings = []
    
    for folder, (code, league) in LEAGUE_FOLDERS.items():
        folder_path = os.path.join(RAW_DIR, folder)
        if not os.path.exists(folder_path):
            continue
        
        for year, season in year_map.items():
            filename = f"{year}{code}.csv"
            path = os.path.join(folder_path, filename)
            if not os.path.exists(path):
                continue
            
            df = pd.read_csv(path)
            df.rename(columns={'Player': 'name', 'Rating': 'rating'}, inplace=True)
            df['season'] = season
            df['league'] = league
            df['name'] = df['name'].str.strip()
            all_ratings.append(df[['name', 'season', 'league', 'rating']])
    
    if not all_ratings:
        return pd.DataFrame()
    
    combined = pd.concat(all_ratings, ignore_index=True)
    combined['name_clean'] = combined['name'].apply(normalise_name)
    return combined

print("Loading ratings...\n")
player_ratings = load_player_ratings()

if not player_ratings.empty:
    print(f"Total: {len(player_ratings):,} | Players: {player_ratings['name'].nunique():,}")
    display(player_ratings.head())

Loading ratings...

Total: 7,263 | Players: 3,101


,name,season,league,rating,name_clean
0,Robert Lewandowski,2020-2021,Bundesliga,8.23,robert lewandowski
1,Erling Haaland,2020-2021,Bundesliga,7.87,erling haaland
2,Jadon Sancho,2020-2021,Bundesliga,7.72,jadon sancho
3,Thomas Müller,2020-2021,Bundesliga,7.70,thomas muller
4,Joshua Kimmich,2020-2021,Bundesliga,7.66,joshua kimmich


In [301]:
master = player_stats.copy()
master['name_clean'] = master['name'].apply(normalise_name)

master['league_comp'] = master['league_comp'].astype(str).str.strip()
master['league_standard'] = master['league_comp'].map(LEAGUE_MAP)

print(f"Base stats: {len(master):,} rows")
master['league_comp'] = master['league_comp'].astype(str).str.strip()
master['league_standard'] = master['league_comp'].map(LEAGUE_MAP)
master['league_comp'] = master['league_comp'].astype(str).str.strip()
master['league_standard'] = master['league_comp'].map(LEAGUE_MAP)


Base stats: 13,636 rows


In [302]:
if not player_ratings.empty:
    print("Merging ratings on [name_clean, season, league]")
    
    master = master.merge(
        player_ratings[['name_clean', 'season', 'league', 'rating']],
        left_on=['name_clean', 'season', 'league_standard'],
        right_on=['name_clean', 'season', 'league'],
        how='left'
    )
    
    master = master.drop(columns=['league'], errors='ignore')
    
    matched = master['rating'].notna().sum()
    print(f"Ratings merged: {matched:,} / {len(master):,} ({matched/len(master)*100:.1f}%)")
else:
    master['rating'] = np.nan
    print("No ratings to merge")

Merging ratings on [name_clean, season, league]
Ratings merged: 6,664 / 13,640 (48.9%)


In [303]:
dupes = master.groupby(['name_clean', 'season']).size()
dupes = dupes[dupes > 1]

if len(dupes) > 0:
    print(f"{len(dupes)} found players appearing multiple times in same season:")
    for (name, season), count in list(dupes.items())[:10]:
        rows = master[(master['name_clean'] == name) & (master['season'] == season)]
        for _, row in rows.iterrows():
            print(f"  {row['name']} | age {row['Age']} | {row['club']} | {season} | rating: {row['rating']}")
    if len(dupes) > 10:
        print(f"and {len(dupes) - 10} more")
else:
    print("No duplicate players in same season")

39 found players appearing multiple times in same season:
  Danilo | age 29 | Juventus | 2020-2021 | rating: 7.34
  Danilo | age 29 | Juventus | 2020-2021 | rating: 6.46
  Danilo | age 31 | Juventus | 2022-2023 | rating: 7.38
  Danilo | age 21 | Nottingham Forest | 2022-2023 | rating: nan
  Danilo | age 32 | Juventus | 2023-2024 | rating: 7.4
  Danilo | age 22 | Nottingham Forest | 2023-2024 | rating: 6.82
  Danilo | age 33 | Juventus | 2024-2025 | rating: nan
  Danilo | age 23 | Nottingham Forest | 2024-2025 | rating: nan
  David López | age 34 | Girona | 2024-2025 | rating: 6.77
  David López | age 21 | Mallorca | 2024-2025 | rating: 6.77
  Diego López | age 20 | Valencia | 2022-2023 | rating: nan
  Diego López | age 40 | Rayo Vallecano | 2022-2023 | rating: nan
  Ederson | age 27 | Manchester City | 2021-2022 | rating: 6.71
  Éderson | age 22 | Salernitana | 2021-2022 | rating: nan
  Ederson | age 28 | Manchester City | 2022-2023 | rating: 6.57
  Éderson | age 23 | Atalanta | 2022-2

In [304]:
master = master.sort_values(['name', 'season']).reset_index(drop=True)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
master.to_csv(OUTPUT_PATH, index=False)

In [305]:
display(master.head(10))

,rank,name,Nation,position,club,league_comp,Age,Born,appearances,Starts,...,CrdR,Gls_90,Ast_90,G+A_90,G-PK_90,G+A-PK_90,season,name_clean,league_standard,rating
0,2700,Aaron Ciammaglichella,it ITA,MF,Torino,it Serie A,19,2005,1,0,...,0,0,0,0,0,0,2024-2025,aaron ciammaglichella,Serie A,NaN
1,1713,Aaron Connolly,ie IRL,FW/MF,Brighton,eng Premier League,20,2000,17,9,...,0,0.23,0.11,0.34,0.23,0.34,2020-2021,aaron connolly,Premier League,NaN
2,2318,Aaron Connolly,ie IRL,FW,Brighton,eng Premier League,21,2000,4,1,...,0,0.00,0.00,0.00,0.00,0.00,2021-2022,aaron connolly,Premier League,NaN
3,67,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,30,1989,36,36,...,0,0.00,0.23,0.23,0.00,0.23,2020-2021,aaron cresswell,Premier League,6.97
4,269,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,31,1989,31,31,...,0,0.07,0.10,0.17,0.07,0.17,2021-2022,aaron cresswell,Premier League,7.10
5,590,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,32,1989,28,24,...,0,0.00,0.04,0.04,0.00,0.04,2022-2023,aaron cresswell,Premier League,6.87
6,1956,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,33,1989,11,4,...,0,0.00,0.00,0.00,0.00,0.00,2023-2024,aaron cresswell,Premier League,NaN
7,1620,Aaron Cresswell,eng ENG,DF,West Ham United,eng Premier League,34,1989,18,10,...,0,0,0,0,0,0,2024-2025,aaron cresswell,Premier League,NaN
8,1729,Aaron Hickey,sct SCO,DF/MF,Bologna,it Serie A,18,2002,11,10,...,1,0.00,0.00,0.00,0.00,0.00,2020-2021,aaron hickey,Serie A,NaN
9,210,Aaron Hickey,sct SCO,MF/DF,Bologna,it Serie A,19,2002,36,34,...,0,0.16,0.03,0.19,0.16,0.19,2021-2022,aaron hickey,Serie A,6.95


In [306]:
print("Position distribution:")
print(master['position'].value_counts().head(20))

Position distribution:
position
MF       4192
DF       3455
FW       1782
MF/FW    1039
GK       1032
FW/MF     811
DF/MF     766
MF/DF     549
DF/FW      12
FW/DF       2
Name: count, dtype: int64


In [307]:
print("Rating coverage by season:")
display(master.groupby('season')['rating'].agg(['count', 'mean', 'min', 'max']))

Rating coverage by season:


,count,mean,min,max
season,,,,
2020-2021,1347,6.750037,5.77,8.35
2021-2022,1328,6.917922,5.71,8.22
2022-2023,1360,6.897176,6.01,8.18
2023-2024,1327,6.900776,6.11,8.12
2024-2025,1302,6.874616,6.03,8.23


In [308]:
master = master.drop(columns=['name_clean', 'league_standard'], errors='ignore')
master = master.sort_values(['name', 'season']).reset_index(drop=True)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
master.to_csv(OUTPUT_PATH, index=False)

print(f"Saved to: {OUTPUT_PATH}")
print(f"Rows: {len(master):,} | Players: {master['name'].nunique():,}")

Saved to: ..\data\processed\player_db.csv
Rows: 13,640 | Players: 5,595
